# 02 - Transform raw cache -> source-of-truth Parquet

Stage B of the phase-5 initial build. Reads the raw OMW cache (produced by
`01_download`), transforms it into source-tagged `concepts` / `lemmas` / `senses`
Parquet under `data/lexicon/` (the source of truth), writes the `_build.json`
manifest, and carves a small sample slice. OMW is the sole source (kaikki was
removed in phase 5.5); a concept's English gloss falls back to the CILI ILI
gloss when OMW has none, tagged `cili`.

Thin caller: all logic is in `ingestion.pipeline.build_initial`. After this runs,
`LexiconStore.from_data_fol` loads the full corpus and the `explore` notebook
under `notebooks/lexicon_corpus/` starts returning rows.


In [ ]:
from loguru import logger as lg

from lang_tools.lexicon.ingestion.pipeline import build_initial
from lang_tools.params.lang_tools_params import get_lang_tools_params

LANGS = [
    "en",
    "pt",
    "es",
    "fr",
    "it",
]
data_fol = get_lang_tools_params().paths.data_fol
lg.info("02-transform: langs={} -> data_fol={}", LANGS, data_fol)
data_fol

## Build

Sources default to the raw cache under `data_fol` (so this needs the `ingest`
extra to read OMW via `wn`, plus the `cili` resource downloaded in `01_download`
for the English-gloss fallback). Pass `extra_manifest=omw_info` (the dict from
`01_download`) to pin the source versions in the manifest.


In [ ]:
lg.info("Building corpus for {} (OMW backbone + CILI fallback, kaikki-free)...", LANGS)
# sample_data_fol carves the committed sample slice into its own corpus at
# data/bootstrap/lexicon/, the folder the store and the webapp tests read.
seed_fol = data_fol / "bootstrap"
summary = build_initial(LANGS, data_fol=data_fol, sample_data_fol=seed_fol)
lg.success("Build complete: counts={} sample={}", summary.counts, summary.sample_counts)
summary.counts, summary.sample_counts

In [ ]:
# Provenance check: confirm the kaikki-free / CILI-fallback state (5.5 Steps 1-2).
# Expect {"omw": ..., "cili": ...} with no "kaikki" key.
from collections import Counter

import pyarrow.parquet as pq

from lang_tools.lexicon.codec import LEXICON_SUBDIR
from lang_tools.lexicon.codec import PROVENANCE_COL

concepts_pq = data_fol / LEXICON_SUBDIR / "concepts.parquet"
provenance = Counter(pq.read_table(concepts_pq).column(PROVENANCE_COL).to_pylist())
lg.info("concept provenance: {}", dict(provenance))
assert "kaikki" not in provenance, "Step 1 guard: no concept may be kaikki-tagged"
dict(provenance)

## Refresh the committed sample seed

The slice carved above is the corpus `data/bootstrap/lexicon/`; the committed
source of truth for it is the JSONL beside it, which `parquetize_seed.ipynb`
and the webapp test fixture read. Export the slice back to JSONL so the two
agree, then commit the JSONL (it lives under the gitignored `data/`, so it
needs `git add -f`, exactly like it already did).

Skip this cell if you are only rebuilding the corpus and do not want to move
the seed.

In [ ]:
from lang_tools.lexicon.corpus import export_table
from lang_tools.lexicon.lemma_store import TABLES

for name in TABLES:
    rows = export_table(name, seed_fol / f"{name}.jsonl", data_fol=seed_fol)
    lg.info("seed {}: {} rows -> {}.jsonl", name, rows, name)

## Spot-check: cross-lingual grouping + gloss coverage


In [ ]:
from lang_tools.lexicon.lemma_store import LexiconStore

store = LexiconStore.from_data_fol(data_fol)

house = next(lem for lem in store.get_lemmas_by_language("en") if lem.text == "house")
concept = store.concepts_for_lemma(house.id)[0]
print("definitions:", concept.definitions)
for lang in LANGS:
    forms = [lem.text for lem in store.lemmas_for_concept(concept.id, language=lang)]
    print(lang, forms)